# SIR + Three-Class MARL-DQN: Rumour Spreading with EGT Node Types

**MARL-DQN framework for modelling rumour propagation and containment on large-scale social networks.**

Key components:
- **Three-Class Node System**: Informed / Uninformed / Indifferent with separate EGT payoff matrices
- **Warm Intensity Curve**: sigmoid shifted so I(0) ≈ 0.25 — rumour can ignite at t=0
- **Superspreader Ignition Window**: forced Share for initially-infected nodes during t < PEAK_T
- **Phase-Split Reward**: early phase rewards spreading (growth), late phase rewards refuting (containment)
- **Proper MARL (CTDE)**: Independent per-agent Q-networks with community-based parameter sharing
- **Peer Pressure**: Transition probability scales with infected-neighbour ratio (threshold contagion)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from collections import deque, namedtuple
from tqdm import trange
import random, os, warnings

warnings.filterwarnings("ignore")

# Global random seed for full reproducibility across PyTorch, NumPy, and Python random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("torch:", torch.__version__, " | PyG:", __import__("torch_geometric").__version__)


## 1. Global Configuration

In [ ]:
# ── Graph & Simulation Scale ─────────────────────────────────────────────
NUM_NODES      = 5000          # network size; large enough for community structure to emerge
MAX_TIME       = 60            # simulation horizon (timesteps)
INIT_INFECTED  = 20            # seed nodes; enough mass for peer-pressure cascades to ignite

# ── DQN Hyperparameters ───────────────────────────────────────────────────
LR             = 1e-3
GAMMA_RL       = 0.99          # discount factor
BATCH_SIZE     = 128           # larger batch stabilises learning on 5000-node graphs
BUFFER_CAP     = 50_000        # replay buffer capacity, scaled for graph size
EPS_START      = 0.90          # initial exploration rate
EPS_END        = 0.05          # minimum exploration rate
EPS_DECAY      = 0.993         # per-episode multiplicative decay
TARGET_UPDATE  = 5             # hard target-network sync interval (episodes)
TRAIN_EPS      = 80            # training episodes; tuned for convergence at this scale

# ── Rumour Intensity Curve (Warm-Floor Growth-then-Fatigue) ───────────────
# I(t) = I_FLOOR + (1 - I_FLOOR) * sigmoid(k*(t-peak)) * exp_decay(t)
# I_FLOOR ensures I(0) ≈ 0.25, so spreading can ignite at t=0 regardless of sigmoid position
PEAK_T         = 20            # novelty peak; sigmoid centre
GROWTH_K       = 0.30          # sigmoid steepness during rise phase
FATIGUE_LAMBDA = 0.05          # exponential decay rate after peak (media fatigue)
I_FLOOR        = 0.25          # warm floor — minimum spreading intensity at any timestep

# ── Peer Pressure & Passive Spread ───────────────────────────────────────
PEER_ALPHA     = 0.70          # maximum adoption probability under full-neighbour infection
BASE_SPREAD    = 0.04          # passive spread floor (independent of peer influence)

# ── Phase-Split Reward Boundary ───────────────────────────────────────────
# Agents are rewarded for spreading before CONTAIN_T, and for refuting after it
CONTAIN_T      = 30            # containment phase start; aligned with outbreak peak

# ── Refutation Mechanics ──────────────────────────────────────────────────
REFUTE_PROB_CONTAIN = 0.40     # P(I→R | Refute action, t >= CONTAIN_T)
REFUTE_PROB_GROWTH  = 0.10     # P(I→R | Refute action, t < CONTAIN_T)
REFUTE_CASCADE      = 0.12     # additional recovery probability per Refuting-R neighbour
SPONTANEOUS_RECOV   = 0.03     # baseline recovery without any Refute action

# ── Three-Class Node Type System ──────────────────────────────────────────
# Node types are fixed at graph creation and shape EGT payoffs throughout simulation
NODE_TYPE_PROBS = [0.30, 0.50, 0.20]   # proportions: Informed / Uninformed / Indifferent

EGT_BASE_PAYOFFS = {
    'Informed':    {'Ignore': -0.10, 'Share': -0.30, 'Refute':  0.40},
    'Uninformed':  {'Ignore':  0.00, 'Share':  0.20, 'Refute':  0.10},
    'Indifferent': {'Ignore':  0.15, 'Share':  0.05, 'Refute':  0.05},
}

# ── MARL Architecture Parameters ─────────────────────────────────────────
N_COMMUNITIES  = 4             # independent agent groups (detected via graph partitioning)
IN_FEATS  = 11                 # node feature dimension (SIR state, degree, PageRank, etc.)
HIDDEN    = 64                 # GRU and GATConv hidden dimension
N_ACTIONS = 3                  # action space: 0=Ignore, 1=Share, 2=Refute

import numpy as _np
_i0 = I_FLOOR + (1 - I_FLOOR) * (1 / (1 + _np.exp(-GROWTH_K * (0 - PEAK_T))))
print('Config loaded.')
print(f'I(0) = {_i0:.3f}  (warm floor guarantee: spreading can ignite at t=0)')
print(f'Containment reward kicks in at t={CONTAIN_T}')


## 2. Model: T-GAT + GRU (AgentGNN)

Each community owns an **independent** AgentGNN.
- `forward()` — GNN path used for both rollout **and** replay (fixes the MLP bypass)
- GATConv captures structural neighbourhood influence
- GRUCell encodes temporal belief history per agent
- `q_head` produces Q-values over {Ignore, Share, Refute}


In [ ]:
class AgentGNN(nn.Module):
    """
    Independent Q-network for one community of agents.

    Architecture:
      GATConv (2-head attention) → GRUCell (temporal belief) → Linear Q-head

    The GNN path is used for BOTH rollout and replay, ensuring the attention
    mechanism is active during training (no linear bypass).
    """
    def __init__(self):
        super().__init__()
        self.gat    = GATConv(IN_FEATS, HIDDEN // 2, heads=2)   # outputs HIDDEN
        self.gru    = nn.GRUCell(HIDDEN, HIDDEN)                 # encodes temporal state
        self.q_head = nn.Linear(HIDDEN, N_ACTIONS)               # Q-values over {Ignore, Share, Refute}

    def forward(self, x, edge_index, h):
        """Full GNN forward — identical code path for action selection and replay updates."""
        x_out  = F.elu(self.gat(x, edge_index))
        h_next = self.gru(x_out, h)
        return self.q_head(h_next), h_next


class MARLSystem:
    """
    Centralised Training, Decentralised Execution (CTDE) MARL system.

    Structure:
      - N_COMMUNITIES independent online + target Q-networks (one per graph community)
      - Community assignment via greedy modularity partitioning (NetworkX)
      - Per-community replay buffers: each community trains on its own agents' experience
      - Decentralised execution: each agent queries only its own community network

    Replay uses proper GAT ego-subgraph reconstruction (not a linear projection bypass).
    Target networks are hard-synced every TARGET_UPDATE episodes across all communities.
    """
    def __init__(self, gc):
        self.gc = gc

        # Partition graph into communities using greedy modularity maximisation
        from networkx.algorithms.community import greedy_modularity_communities
        communities = list(greedy_modularity_communities(gc.G))
        self.node_community = np.zeros(gc.n, dtype=np.int64)
        for c_idx, community in enumerate(communities):
            for node in community:
                self.node_community[node] = min(c_idx, N_COMMUNITIES - 1)
        n_actual = min(len(communities), N_COMMUNITIES)
        print(f'  Communities detected: {len(communities)} → using {n_actual} independent networks')

        # One online + target network and replay buffer per community
        self.online  = [AgentGNN() for _ in range(N_COMMUNITIES)]
        self.target  = [AgentGNN() for _ in range(N_COMMUNITIES)]
        self.opts    = [torch.optim.Adam(net.parameters(), lr=LR) for net in self.online]
        self.buffers = [ReplayBuffer() for _ in range(N_COMMUNITIES)]

        for i in range(N_COMMUNITIES):
            self.target[i].load_state_dict(self.online[i].state_dict())
            self.target[i].eval()

        # Pre-index community membership for fast per-community masking
        self.community_masks = [
            np.where(self.node_community == c)[0] for c in range(N_COMMUNITIES)
        ]

    def select_actions(self, x, h, eps):
        """
        Decentralised execution: each community network acts for its own agents.
        ε-greedy exploration is applied independently per community.
        Returns: actions array (n,), updated hidden states h_new (n, HIDDEN).
        """
        actions = np.zeros(self.gc.n, dtype=np.int64)
        h_new   = torch.zeros_like(h)
        with torch.no_grad():
            for c in range(N_COMMUNITIES):
                mask = self.community_masks[c]
                if len(mask) == 0:
                    continue
                # Full graph context for GNN; only this community's nodes act
                q_all, h_all = self.online[c](x, self.gc.ei, h)
                q_c = q_all[mask]
                greedy  = q_c.argmax(dim=1).numpy()
                explore = np.random.rand(len(mask)) < eps
                actions[mask] = np.where(explore,
                    np.random.randint(0, N_ACTIONS, len(mask)), greedy)
                h_new[mask] = h_all[mask]
        return actions, h_new

    def _build_ego_batch(self, batch_transitions, use_next_state=False):
        """
        Reconstruct 1-hop ego-subgraph for each transition in the replay batch.

        Each node's ego graph (center + 1-hop neighbours) is re-indexed locally
        so GAT attention operates on the correct neighbourhood structure.
        Neighbour features use cached static values (degree, PageRank, node_type)
        with neutral SIR state — standard neighbourhood-sampling approximation.

        Returns:
          x_batch  : [N_total, IN_FEATS] concatenated ego-graph features
          ei_batch : [2, E_total]        concatenated edge indices (globally offset)
          centers  : LongTensor[B]       global position of each center node in x_batch
        """
        feats   = []
        edges   = []
        centers = []
        offset  = 0
        for tr in batch_transitions:
            i        = tr.node_idx
            ego_n    = self.gc.ego_nodes[i]
            ei_local = self.gc.ego_edges[i]
            k        = len(ego_n)

            ego_x = np.zeros((k, IN_FEATS), dtype=np.float32)
            # Center node uses its stored feature vector (state s or ns)
            ego_x[0] = tr.ns if use_next_state else tr.s
            # Neighbours use cached static features with neutral SIR state
            for k_local in range(1, k):
                orig = ego_n[k_local]
                ego_x[k_local, 1]  = self.gc.deg[orig]
                ego_x[k_local, 4]  = self.gc.pr[orig]
                ego_x[k_local, 10] = self.gc.node_types[orig] / 2.0

            feats.append(torch.from_numpy(ego_x))
            edges.append(ei_local + offset)
            centers.append(offset)
            offset += k

        x_batch  = torch.cat(feats, dim=0)
        ei_batch = torch.cat(edges, dim=1) if edges else torch.zeros(2, 0, dtype=torch.long)
        centers  = torch.tensor(centers, dtype=torch.long)
        return x_batch, ei_batch, centers

    def update(self, c):
        """
        DQN update for community c using proper GAT ego-subgraph replay.

        Samples a minibatch, reconstructs each agent's ego graph, runs the
        full GATConv forward, and applies double-Q Bellman update with
        Huber loss and gradient clipping for training stability.
        """
        buf = self.buffers[c]
        if len(buf) < BATCH_SIZE:
            return 0.0

        batch = buf.sample(BATCH_SIZE)
        a_b   = torch.tensor([b.a for b in batch], dtype=torch.long)
        r_b   = torch.tensor([b.r for b in batch], dtype=torch.float32)
        d_b   = torch.tensor([b.d for b in batch], dtype=torch.float32)

        net_c = self.online[c]
        tgt_c = self.target[c]

        # Build ego-subgraph batches for current (s) and next (ns) states
        x_s,  ei_s,  cen_s  = self._build_ego_batch(batch, use_next_state=False)
        x_ns, ei_ns, cen_ns = self._build_ego_batch(batch, use_next_state=True)

        h_zero_s  = torch.zeros(x_s.size(0),  HIDDEN)
        h_zero_ns = torch.zeros(x_ns.size(0), HIDDEN)

        # Online network: Q(s, a) for taken actions
        q_all_s, _ = net_c(x_s, ei_s, h_zero_s)
        q_pred     = q_all_s[cen_s].gather(1, a_b.unsqueeze(1)).squeeze(1)

        # Target network: max_a' Q_target(s', a') for Bellman target
        with torch.no_grad():
            q_all_ns, _ = tgt_c(x_ns, ei_ns, h_zero_ns)
            q_next      = q_all_ns[cen_ns].max(1)[0]

        q_target = r_b + GAMMA_RL * q_next * (1.0 - d_b)

        loss = F.smooth_l1_loss(q_pred, q_target)
        self.opts[c].zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.online[c].parameters(), 1.0)  # gradient clipping
        self.opts[c].step()
        return loss.item()

    def sync_targets(self):
        """Hard-copy online → target weights for all community networks simultaneously."""
        for c in range(N_COMMUNITIES):
            self.target[c].load_state_dict(self.online[c].state_dict())


print('AgentGNN + MARLSystem defined.')
_m = AgentGNN()
_x = torch.randn(10, IN_FEATS)
_ei = torch.zeros(2, 0, dtype=torch.long)
print(f'  AgentGNN parameter count: {sum(p.numel() for p in _m.parameters()):,}')


## 3. Replay Buffer

In [ ]:
Transition = namedtuple("Transition", ["s", "a", "r", "ns", "d", "node_idx"])

class ReplayBuffer:
    """
    Experience replay buffer that stores node_idx alongside transition data.

    node_idx is required at training time to reconstruct the node's 1-hop ego-subgraph,
    enabling proper GAT attention during replay (as opposed to a linear projection bypass).
    """
    def __init__(self, cap=BUFFER_CAP):
        self.buf = deque(maxlen=cap)

    def push(self, s, a, r, ns, d, node_idx):
        self.buf.append(Transition(
            np.asarray(s, np.float32), int(a), float(r),
            np.asarray(ns, np.float32), float(d), int(node_idx)
        ))

    def sample(self, n):
        return random.sample(self.buf, n)

    def __len__(self):
        return len(self.buf)

print("ReplayBuffer defined (stores node_idx for ego-subgraph GAT replay).")


## 4. Graph Helpers & Feature Extraction

In [ ]:
def make_graph(gtype='ba', n=NUM_NODES):
    """Generate a synthetic graph of the specified topology type."""
    if gtype == 'ba':   return nx.barabasi_albert_graph(n, 3, seed=SEED)
    elif gtype == 'er': return nx.erdos_renyi_graph(n, 6/n, seed=SEED)
    elif gtype == 'ws': return nx.watts_strogatz_graph(n, k=6, p=0.1, seed=SEED)
    elif gtype == 'sbm':
        q = n // 4
        sizes = [q, q, q, n - 3*q]
        probs = [[0.02,.002,.002,.002],[.002,.02,.002,.002],
                 [.002,.002,.02,.002],[.002,.002,.002,.02]]
        return nx.stochastic_block_model(sizes, probs, seed=SEED)
    raise ValueError(gtype)


def build_edge_index(G):
    """Convert NetworkX graph edges to PyTorch Geometric COO edge_index (bidirectional)."""
    e = list(G.edges())
    r = [(b, a) for a, b in e]
    return torch.tensor(list(zip(*(e + r))), dtype=torch.long)


def rumour_intensity(t):
    """
    Warm-floor Growth-then-Fatigue intensity curve.

    I(t) = I_FLOOR + (1 - I_FLOOR) * sigmoid(k*(t-peak)) * exp(-λ*max(0,t-peak))

    The warm floor (I_FLOOR=0.25) ensures spreading ignites at t=0 regardless
    of where the sigmoid sits — critical for modelling real-world viral outbreaks
    that begin spreading immediately.
    """
    rise    = 1.0 / (1.0 + np.exp(-GROWTH_K * (t - PEAK_T)))
    fatigue = np.exp(-FATIGUE_LAMBDA * max(0, t - PEAK_T))
    return float(I_FLOOR + (1.0 - I_FLOOR) * rise * fatigue)


# Node-type name mapping and EGT payoff matrix (shape [3, 3]: type × action)
_TYPE_NAMES = ['Informed', 'Uninformed', 'Indifferent']
_TYPE_PAYOFFS = np.array([
    [EGT_BASE_PAYOFFS[t][a] for a in ['Ignore', 'Share', 'Refute']]
    for t in _TYPE_NAMES
], dtype=np.float32)


def assign_node_types(n):
    """Sample node types (0=Informed, 1=Uninformed, 2=Indifferent) at graph creation."""
    return np.random.choice(3, size=n, p=NODE_TYPE_PROBS).astype(np.int64)


def egt_payoff(node_types, actions, peer_ratio, intensity):
    """
    Compute EGT-modulated payoff for each node given its type, action, and social context.

    Modifiers applied on top of the base payoff matrix:
      - Informed + Share:      credibility cost scales with current intensity
      - Informed + Refute:     truth-telling bonus amplified by local peer pressure
      - Uninformed + Share:    social-fit reward scales with infected-neighbour ratio
      - Indifferent (any):     all payoffs attenuated by peer pressure (models inertia)

    Returns payoff array of shape (n,).
    """
    base    = _TYPE_PAYOFFS[node_types, actions]
    payoffs = base.copy()

    for i in range(len(node_types)):
        t = node_types[i]
        a = actions[i]
        if t == 0:    # Informed
            if a == 1: payoffs[i] -= 0.3 * intensity
            elif a == 2: payoffs[i] += 0.2 * peer_ratio[i]
        elif t == 1:  # Uninformed
            if a == 1: payoffs[i] += 0.3 * peer_ratio[i]
            elif a == 2: payoffs[i] += 0.05
        elif t == 2:  # Indifferent — resists social pressure
            payoffs[i] *= (1.0 - 0.5 * peer_ratio[i])

    return payoffs


class GraphCache:
    """
    Pre-computes and caches all graph-level quantities used repeatedly during training:
    edge_index, PageRank, normalised degree, neighbour lists, node types,
    and per-node 1-hop ego-subgraph edge indices (for GAT replay).
    """
    def __init__(self, G):
        self.G    = G
        self.n    = G.number_of_nodes()
        self.ei   = build_edge_index(G)
        pr_dict   = nx.pagerank(G)
        self.pr   = np.array([pr_dict[i]         for i in range(self.n)], np.float32)
        self.deg  = np.array([G.degree(i)/self.n  for i in range(self.n)], np.float32)
        self.nbrs = [list(G.neighbors(i))          for i in range(self.n)]
        # Node types are fixed for the lifetime of the graph (stable across episodes)
        self.node_types = assign_node_types(self.n)

        # Pre-compute 1-hop ego-subgraph edge_index per node (relabelled locally).
        # Stored so replay can run the full GAT forward on the correct neighbourhood
        # without accessing the full graph at replay time.
        self.ego_edges = []
        self.ego_nodes = []
        for i in range(self.n):
            nbrs_i = self.nbrs[i]
            ego_n  = [i] + list(nbrs_i)          # center node is always local index 0
            local  = {orig: k for k, orig in enumerate(ego_n)}
            if len(nbrs_i) > 0:
                ei = []
                for nb in nbrs_i:
                    ei.append([local[i],  local[nb]])
                    ei.append([local[nb], local[i]])
                ei_t = torch.tensor(ei, dtype=torch.long).T
            else:
                ei_t = torch.zeros(2, 0, dtype=torch.long)
            self.ego_edges.append(ei_t)
            self.ego_nodes.append(np.asarray(ego_n, dtype=np.int64))

    def features(self, states, t):
        """
        Build (n × 11) node feature matrix for timestep t.

        Features: [own SIR state, degree centrality, infected-nbr ratio,
                   global intensity, PageRank, mean-nbr state, refuted-nbr ratio,
                   susceptible-nbr ratio, peer pressure ratio (copy), intensity copy,
                   normalised node type]
        """
        n         = self.n
        intensity = rumour_intensity(t)
        ir = np.array([sum(states[j]==1 for j in self.nbrs[i]) / (len(self.nbrs[i])+1)
                       for i in range(n)], np.float32)
        rr = np.array([sum(states[j]==2 for j in self.nbrs[i]) / (len(self.nbrs[i])+1)
                       for i in range(n)], np.float32)
        sr = np.array([sum(states[j]==0 for j in self.nbrs[i]) / (len(self.nbrs[i])+1)
                       for i in range(n)], np.float32)
        nm = np.array([float(np.mean([states[j] for j in self.nbrs[i]]))
                       if self.nbrs[i] else 0. for i in range(n)], np.float32)
        return torch.tensor(np.stack([
            states.astype(np.float32),                # [0]  own SIR state (0/1/2)
            self.deg,                                  # [1]  normalised degree centrality
            ir,                                        # [2]  infected-neighbour ratio
            np.full(n, intensity, np.float32),         # [3]  global rumour intensity I(t)
            self.pr,                                   # [4]  PageRank score
            nm,                                        # [5]  mean neighbour SIR state
            rr,                                        # [6]  refuted-neighbour ratio
            sr,                                        # [7]  susceptible-neighbour ratio
            ir,                                        # [8]  peer pressure proxy (= [2])
            np.full(n, intensity, np.float32),         # [9]  intensity duplicate (local copy)
            self.node_types.astype(np.float32) / 2.0, # [10] normalised node type (0/0.5/1)
        ], axis=1))


print('Graph helpers + rumour_intensity + three-class node types defined.')

# Plot the warm-floor intensity curve to verify I(0) > 0
ts = np.arange(MAX_TIME)
ri = [rumour_intensity(t) for t in ts]
plt.figure(figsize=(8, 3))
plt.plot(ts, ri, 'r-', linewidth=2)
plt.axvline(PEAK_T,  color='gray',   linestyle='--', label=f'Novelty peak t={PEAK_T}')
plt.axhline(ri[0],  color='orange', linestyle=':',  label=f'I(0)={ri[0]:.2f} (warm start)')
plt.title(f'Rumour Intensity Curve  [I_FLOOR={I_FLOOR}, PEAK_T={PEAK_T}, λ={FATIGUE_LAMBDA}]')
plt.xlabel('Time'); plt.ylabel('Intensity I(t)'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'I(0)={ri[0]:.3f}  |  peak I={max(ri):.3f} at t={ts[np.argmax(ri)]}  |  I(CONTAIN_T)={ri[CONTAIN_T]:.3f}')


## 4b. Three-Class Node Type System

Each node is assigned one of three classes at graph creation time:

| Class | Proportion | Behaviour |
|---|---|---|
| **Informed** | 30% | Critical evaluators — low susceptibility, high refutation payoff |
| **Uninformed** | 50% | Social conformists — high susceptibility, Share payoff scales with peer pressure |
| **Indifferent** | 20% | Inert majority — low susceptibility, default to Ignore (comfort payoff) |

The Indifferent class creates the **plateau** in the middle of the outbreak — the rumour slows but doesn't die — which produces the realistic bell curve shape.


In [ ]:
# ── Visualise three-class node type distribution on a sample graph ───
G_sample = make_graph('ba', n=NUM_NODES)
gc_sample = GraphCache(G_sample)
nt = gc_sample.node_types

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: bar chart of type proportions
counts = [sum(nt==0), sum(nt==1), sum(nt==2)]
labels = ['Informed', 'Uninformed', 'Indifferent']
colors = ['#2196F3', '#FF5722', '#9E9E9E']
axes[0].bar(labels, counts, color=colors, edgecolor='black', alpha=0.85)
for j, (lbl, cnt) in enumerate(zip(labels, counts)):
    axes[0].text(j, cnt+1, f'{cnt}\n({cnt/NUM_NODES*100:.0f}%)', ha='center', fontsize=10)
axes[0].set_title('Node Type Distribution (BA graph, n=300)')
axes[0].set_ylabel('Count'); axes[0].grid(axis='y', alpha=0.3)

# Right: EGT payoff matrix heatmap
payoff_data = np.array([
    [EGT_BASE_PAYOFFS[t][a] for a in ['Ignore','Share','Refute']]
    for t in ['Informed','Uninformed','Indifferent']
])
im = axes[1].imshow(payoff_data, cmap='RdYlGn', aspect='auto', vmin=-0.4, vmax=0.5)
axes[1].set_xticks([0,1,2]); axes[1].set_xticklabels(['Ignore','Share','Refute'])
axes[1].set_yticks([0,1,2]); axes[1].set_yticklabels(['Informed','Uninformed','Indifferent'])
axes[1].set_title('EGT Base Payoff Matrix\n(scaled by peer_ratio & intensity at runtime)')
for row in range(3):
    for col in range(3):
        axes[1].text(col, row, f'{payoff_data[row,col]:+.2f}', ha='center', va='center',
                    fontsize=12, fontweight='bold',
                    color='white' if abs(payoff_data[row,col]) > 0.25 else 'black')
plt.colorbar(im, ax=axes[1], label='Base Payoff')

fig.tight_layout(); plt.show()
print(f'Node types: Informed={counts[0]}, Uninformed={counts[1]}, Indifferent={counts[2]}')
print('EGT payoffs: Indifferent nodes create the mid-outbreak plateau (realistic bell curve)')


## 5. Environment: SIR Transition & Reward Function

**FIX 1 — Warm intensity**: I(0) ≈ 0.25 so spreading ignites immediately.

**FIX 2 — Superspreader ignition window**: for `t < PEAK_T`, initially-infected nodes
force `Share` regardless of learned policy — models the real-world dynamic where early
spreaders are highly motivated before containment kicks in.

**FIX 3 — Node-type susceptibility**: Uninformed (100%), Informed (50%), Indifferent (20%).
Adoption probability = `PEER_ALPHA × infected_ratio × intensity × type_susceptibility + BASE_SPREAD`.

**FIX 4 — Phase-split reward**:
```
t < PEAK_T  (GROWTH):   reward spreading  — lets outbreak develop naturally
t >= PEAK_T (CONTAIN):  reward refuting   — agents learn containment after peak
```
EGT payoff per (node_type × action) scales reward magnitude throughout.


In [ ]:
def env_step(states, actions, gc, t, init_nodes=None):
    """
    SIR state transition with peer-pressure contagion and learned refutation.

    Spreading mechanics (active in all phases):
      - Peer pressure: P(S→I) scales with infected-neighbour ratio × intensity × node susceptibility
      - Passive infection: baseline spread even when agent Ignores (no active choice)
      - Superspreader window: seed nodes are forced to Share for t < PEAK_T

    Containment mechanics (active when t >= CONTAIN_T):
      - Refute action: P(I→R) = 40% vs 3% spontaneous — agents can actively accelerate recovery
      - Refutation cascade: each Refuting recovered neighbour adds 12% additional recovery pressure
      - Social inoculation: Susceptible nodes choosing Refute reduce their own spread probability by 65%
    """
    intensity  = rumour_intensity(t)
    new        = states.copy()
    node_types = gc.node_types
    in_contain = (t >= CONTAIN_T)

    # Superspreader ignition: force seed nodes to Share during the novelty window
    if init_nodes is not None and t < PEAK_T:
        actions = actions.copy()
        for i in init_nodes:
            if states[i] == 1:
                actions[i] = 1

    # Type-susceptibility multipliers: Uninformed nodes are maximally susceptible
    TYPE_SUSCEPT = np.array([0.55, 1.00, 0.25], dtype=np.float32)

    # Refutation cascade: count Refuting-R neighbours for each node
    refuting_R_nbrs = np.array([
        sum(1 for j in gc.nbrs[i] if states[j] == 2 and actions[j] == 2)
        for i in range(gc.n)
    ], dtype=np.float32)

    for i in range(gc.n):
        nbrs_i = gc.nbrs[i]
        n_nbrs = len(nbrs_i) + 1
        suscep = TYPE_SUSCEPT[node_types[i]]

        if states[i] == 0:    # Susceptible node
            infected_ratio = sum(states[j] == 1 for j in nbrs_i) / n_nbrs
            p_peer   = PEER_ALPHA * infected_ratio * intensity * suscep
            p_spread = BASE_SPREAD * suscep + p_peer
            # Social inoculation: Refute action reduces spread probability in containment phase
            if actions[i] == 2 and in_contain:
                p_spread *= 0.35
            if   actions[i] == 1 and np.random.rand() < p_spread: new[i] = 1
            elif actions[i] != 1 and np.random.rand() < p_peer * 0.4: new[i] = 1

        elif states[i] == 1:  # Infected / spreading node
            if in_contain:
                # Containment phase: Refute dramatically accelerates recovery
                p_refute = REFUTE_PROB_CONTAIN if actions[i] == 2 else SPONTANEOUS_RECOV
                p_refute = min(0.95, p_refute + REFUTE_CASCADE * refuting_R_nbrs[i] / n_nbrs)
            else:
                # Growth phase: early refutation is possible but weak
                p_refute = REFUTE_PROB_GROWTH if (actions[i] == 2 and t > 5) else SPONTANEOUS_RECOV
            if np.random.rand() < p_refute:
                new[i] = 2

    # Residual passive spread for Susceptible nodes not yet updated
    for i in range(gc.n):
        if states[i] == 0 and new[i] == 0:
            suscep = TYPE_SUSCEPT[node_types[i]]
            resist = 0.35 if (actions[i] == 2 and in_contain) else 1.0
            for nb in gc.nbrs[i]:
                if states[nb] == 1 and np.random.rand() < BASE_SPREAD * intensity * suscep * resist:
                    new[i] = 1
                    break

    return new


def reward_fn(states, actions, gc, t, flat=False):
    """
    Phase-split reward function.

    Growth phase  (t < CONTAIN_T): spreading is rewarded — lets the outbreak develop naturally.
    Contain phase (t >= CONTAIN_T): Refute yields large positive reward; Share yields large penalty.
    The reward gap between phases must dominate EGT noise to drive a clear policy transition.

    EGT scaling: each node's raw reward is multiplied by (1 + 0.5 * egt_payoff),
    so node type modulates reward magnitude without overriding phase structure.

    flat=True: returns simple ±1 signal (used for No-EGT ablation condition).
    """
    intensity  = rumour_intensity(t)
    r          = np.zeros(gc.n, np.float32)
    node_types = gc.node_types
    in_contain = (t >= CONTAIN_T)

    if flat:
        for i in range(gc.n):
            if   states[i] == 1 and actions[i] == 2: r[i] =  1.0
            elif states[i] == 0 and actions[i] == 1: r[i] = -1.0
        return r

    pr  = np.array([sum(states[j]==1 for j in gc.nbrs[i]) / (len(gc.nbrs[i])+1)
                    for i in range(gc.n)], np.float32)
    egt = egt_payoff(node_types, actions, pr, intensity)

    for i in range(gc.n):
        a = actions[i]

        if states[i] == 0:    # Susceptible
            if not in_contain:
                r[i] = 0.15 if a == 1 else 0.0
            else:
                if   a == 1: r[i] = -2.0 * intensity
                elif a == 2: r[i] =  0.5 + 0.4 * (1.0 - pr[i])
                else:        r[i] =  0.1

        elif states[i] == 1:  # Infected
            if not in_contain:
                r[i] = 0.4 * intensity if a == 1 else (-0.1 if a == 2 else 0.0)
            else:
                if   a == 2: r[i] =  4.0                       # strong containment reward
                elif a == 1: r[i] = -2.0 * intensity * pr[i]   # penalise spreading when contained
                else:        r[i] = -0.3

        else:                 # Recovered
            # Recovered nodes that continue refuting sustain the cascade effect
            r[i] = 0.3 if (a == 2 and in_contain) else 0.05

        r[i] *= (1.0 + 0.5 * egt[i])   # EGT-modulated reward scaling

    return r


print('env_step + reward_fn defined.')
print(f'CONTAIN_T={CONTAIN_T}: Refute P(I→R)={REFUTE_PROB_CONTAIN} vs spontaneous={SPONTANEOUS_RECOV}')
print(f'Refutation cascade per Refuting-R neighbour: +{REFUTE_CASCADE}')
print(f'Social inoculation (Susceptible+Refute): p_spread × 0.35')


## 6. Classical SIR (baseline — no learning)

In [ ]:
def run_sir(beta=0.2, gamma=0.02, init_infected=INIT_INFECTED,
            max_time=MAX_TIME, gtype='ba'):
    """
    Classical SIR baseline without any learned policy.

    Uses the same rumour_intensity curve as MARL for a fair comparison —
    spread probability scales with I(t) rather than being fixed at β.
    This ensures any performance gap reflects policy quality, not intensity mismatch.
    """
    G      = make_graph(gtype)
    n      = G.number_of_nodes()
    states = np.zeros(n, dtype=np.int64)
    states[np.random.choice(n, init_infected, replace=False)] = 1
    hist   = {'S': [], 'I': [], 'R': []}

    for t in range(max_time):
        intensity = rumour_intensity(t)
        new = states.copy()
        for u in np.where(states == 1)[0]:
            for v in G.neighbors(u):
                if states[v] == 0 and np.random.rand() < beta * intensity:
                    new[v] = 1
            if np.random.rand() < gamma:
                new[u] = 2
        states = new
        hist['S'].append(int(np.sum(states == 0)))
        hist['I'].append(int(np.sum(states == 1)))
        hist['R'].append(int(np.sum(states == 2)))

    return hist

print('Classical SIR baseline (intensity-modulated) defined.')


## 7. Proper MARL Training Loop

**What makes this proper MARL (CTDE — Centralised Training, Decentralised Execution):**
- `N_COMMUNITIES` **independent** online + target networks (one per agent group)
- Community assignment via **greedy modularity** graph partitioning
- Each community has its **own replay buffer** — agents train on their own group's experience
- ε-greedy exploration **per community** with shared decay schedule
- GNN path used for action selection; GRU-projected features for batched replay
- **Decentralised execution**: each agent queries only its community network
- Hard target sync across **all** community networks simultaneously


In [ ]:
def train_marl(gtype='ba', init_infected=INIT_INFECTED,
               n_eps=TRAIN_EPS, flat_reward=False, verbose=True, label=''):
    """
    CTDE MARL training loop with N_COMMUNITIES independent networks.

    Per episode:
      1. Initialise SIR state (random seed set of INIT_INFECTED nodes)
      2. For each timestep: decentralised action selection → environment step →
         store transitions in community replay buffers → update each community network
      3. Hard-sync target networks every TARGET_UPDATE episodes
      4. Decay ε for all communities simultaneously

    Returns trained MARLSystem and training diagnostics (reward, loss, final_I curves).
    """
    gc    = GraphCache(make_graph(gtype))
    marl  = MARLSystem(gc)
    eps   = EPS_START
    SUBSAMPLE = max(1, gc.n // 50)   # subsample to limit buffer writes per step

    curves = {'rewards': [], 'losses': [], 'final_I': []}
    desc   = f'Training MARL [{gtype.upper()}{" | "+label if label else ""}]'

    for ep in (trange(n_eps, desc=desc) if verbose else range(n_eps)):
        states     = np.zeros(gc.n, dtype=np.int64)
        init_nodes = np.random.choice(gc.n, init_infected, replace=False)
        states[init_nodes] = 1
        h          = torch.zeros(gc.n, HIDDEN)
        ep_reward  = 0.0
        ep_loss    = 0.0
        loss_steps = 0

        for t in range(MAX_TIME):
            x = gc.features(states, t)

            # Decentralised action selection (each community acts independently)
            actions, h_new = marl.select_actions(x, h, eps)

            # Environment step with superspreader ignition window
            new_states = env_step(states, actions, gc, t, init_nodes=init_nodes)
            rewards    = reward_fn(states, actions, gc, t, flat=flat_reward)
            x_next     = gc.features(new_states, t + 1)
            done       = float(t == MAX_TIME - 1)

            # Store transitions — each node goes into its community's buffer
            for i in range(0, gc.n, SUBSAMPLE):
                c = marl.node_community[i]
                marl.buffers[c].push(
                    x[i].numpy(), actions[i], rewards[i],
                    x_next[i].numpy(), done, node_idx=i
                )

            # Independent DQN update for each community
            for c in range(N_COMMUNITIES):
                loss_c = marl.update(c)
                ep_loss += loss_c
                loss_steps += 1

            ep_reward += float(rewards.mean())
            states = new_states
            h      = h_new.detach()

        # Hard-sync all target networks simultaneously
        if (ep + 1) % TARGET_UPDATE == 0:
            marl.sync_targets()

        eps = max(EPS_END, eps * EPS_DECAY)
        curves['rewards'].append(ep_reward)
        curves['losses'].append(ep_loss / max(1, loss_steps))
        curves['final_I'].append(int(np.sum(states == 1)))

    return marl, curves


def eval_marl(marl_system, gtype='ba', init_infected=INIT_INFECTED, max_time=MAX_TIME):
    """
    Greedy rollout of a trained MARL system (ε=0, decentralised execution).
    Returns SIR history dict with S/I/R counts per timestep.
    """
    gc         = marl_system.gc
    states     = np.zeros(gc.n, dtype=np.int64)
    init_nodes = np.random.choice(gc.n, init_infected, replace=False)
    states[init_nodes] = 1
    h          = torch.zeros(gc.n, HIDDEN)
    hist       = {'S': [], 'I': [], 'R': []}

    for net in marl_system.online:
        net.eval()

    with torch.no_grad():
        for t in range(max_time):
            x           = gc.features(states, t)
            actions, h  = marl_system.select_actions(x, h, eps=0.0)
            states      = env_step(states, actions, gc, t, init_nodes=init_nodes)
            hist['S'].append(int(np.sum(states == 0)))
            hist['I'].append(int(np.sum(states == 1)))
            hist['R'].append(int(np.sum(states == 2)))

    return hist


def eval_marl_avg(marl_system, n_evals=5, init_infected=INIT_INFECTED, max_time=MAX_TIME):
    """
    Average MARL evaluation over n_evals independent rollouts with different seed sets.

    Reduces evaluation variance — reported numbers reflect trained policy quality
    rather than a single lucky or unlucky initial seed configuration.
    Returns mean S/I/R curves plus standard deviation for I and R.
    """
    rollouts = [eval_marl(marl_system, init_infected=init_infected, max_time=max_time)
                for _ in range(n_evals)]
    I_stack = np.array([r['I'] for r in rollouts])
    R_stack = np.array([r['R'] for r in rollouts])
    S_stack = np.array([r['S'] for r in rollouts])
    return {
        'S':       S_stack.mean(axis=0).tolist(),
        'I':       I_stack.mean(axis=0).tolist(),
        'R':       R_stack.mean(axis=0).tolist(),
        'std_I':   I_stack.std(axis=0, ddof=1).tolist() if n_evals > 1 else [0.0]*max_time,
        'std_R':   R_stack.std(axis=0, ddof=1).tolist() if n_evals > 1 else [0.0]*max_time,
        'n_evals': n_evals,
    }


print('train_marl + eval_marl + eval_marl_avg defined.')


## 8. Plotting & Metric Helpers

In [ ]:
def plot_sir_pair(hs, hm, title="", fname=None):
    """Side-by-side SIR curves: classical SIR (left) vs MARL-DQN (right) with containment annotations."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, h, lbl in zip(axes, [hs, hm],
                           ["Classical SIR (fixed β)", "MARL-DQN (learned policy)"]):
        ax.plot(h["S"], "b-", label="S")
        ax.plot(h["I"], "r-", label="I")
        ax.plot(h["R"], "g-", label="R")
        ax.set_title(f"{lbl}{title}"); ax.set_xlabel("Time"); ax.set_ylabel("Nodes")
        ax.legend(); ax.grid(alpha=0.3)
    axes[1].axvline(CONTAIN_T, color='black', linestyle='--', linewidth=1.5,
                    label=f'Contain phase (t={CONTAIN_T})')
    axes[1].axvspan(CONTAIN_T, MAX_TIME, alpha=0.06, color='green', label='Containment region')
    axes[1].legend()
    axes[1].annotate('Learned\ncontainment', xy=(CONTAIN_T+1, 10), xytext=(CONTAIN_T+8, 40),
                     arrowprops=dict(arrowstyle='->', color='darkgreen'),
                     color='darkgreen', fontsize=9)
    fig.tight_layout(); plt.show()
    if fname: fig.savefig(f"outputs/{fname}.png", dpi=110, bbox_inches="tight")


def plot_training_curves(curves, fname=None):
    """Visualise training convergence: episode reward, Huber loss, and final infected count."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(curves["rewards"]);            axes[0].set_title("Episode Reward")
    axes[1].plot(curves["losses"], "orange");   axes[1].set_title("TD Loss (Huber)")
    axes[2].plot(curves["final_I"], "crimson"); axes[2].set_title("Final Infected ↓")
    for ax in axes:
        ax.set_xlabel("Episode"); ax.grid(alpha=0.3)
    fig.suptitle("DQN Training Convergence", fontsize=12, fontweight="bold")
    fig.tight_layout(); plt.show()
    if fname: fig.savefig(f"outputs/{fname}.png", dpi=110, bbox_inches="tight")


def plot_multi_I(results, title, fname=None):
    """Overlay multiple infected curves for ablation or sweep comparisons."""
    fig, ax = plt.subplots(figsize=(10, 5))
    for lbl, h in results:
        ax.plot(h["I"], label=lbl)
    ax.set_title(title); ax.set_xlabel("Time"); ax.set_ylabel("Infected")
    ax.legend(); ax.grid(alpha=0.3); fig.tight_layout(); plt.show()
    if fname: fig.savefig(f"outputs/{fname}.png", dpi=110, bbox_inches="tight")


def glob_pol(hist):
    """Compute per-timestep global polarization index: |I - R| / (I + R)."""
    I = np.array(hist["I"]); R = np.array(hist["R"])
    return np.abs(I - R) / (I + R + 1e-9)


def summarize(hist):
    """Summarise an SIR history into key scalar metrics."""
    I = np.array(hist["I"]); R = np.array(hist["R"])
    return {"peak_I": int(np.max(I)), "t_peak": int(np.argmax(I)),
            "final_R": int(R[-1]),    "avg_I": float(np.mean(I))}


def print_table(title, results):
    print(f"{'='*62}{title}{'='*62}")
    print(f"{'Label':<32} {'peak_I':>7} {'t_peak':>7} {'final_R':>8} {'avg_I':>8}")
    print("-"*62)
    for lbl, h in results:
        m = summarize(h)
        print(f"{lbl:<32} {m['peak_I']:>7} {m['t_peak']:>7} {m['final_R']:>8} {m['avg_I']:>8.1f}")


os.makedirs("outputs", exist_ok=True)


def influential_efficiency(gc, hist_marl, hist_sir, top_k_frac=0.10):
    """
    Influential Efficiency (IE): measures whether MARL leverages high-PageRank nodes
    better than classical SIR during the containment phase.

    Definition:
      IE = (hub_recovery_rate_MARL − hub_recovery_rate_SIR) / (hub_recovery_rate_SIR + ε)

    Hub nodes are the top-k% by PageRank. Hub recovery rate is computed as
    mean(R/(R+I)) over t >= CONTAIN_T, weighted by the PageRank mass in the top-k nodes.
    IE > 0 indicates MARL recovers influential nodes faster than classical SIR.
    """
    n     = gc.n
    top_k = max(1, int(n * top_k_frac))

    # Hub nodes: top-k% by PageRank (most structurally influential)
    hub_nodes = set(np.argsort(gc.pr)[-top_k:].tolist())

    I_m = np.array(hist_marl['I']); R_m = np.array(hist_marl['R'])
    I_s = np.array(hist_sir['I']);  R_s = np.array(hist_sir['R'])

    # PageRank mass concentrated in top-k nodes (structural influence weight)
    pr_sorted     = np.sort(gc.pr)[::-1]
    pr_top_weight = pr_sorted[:top_k].sum() / (gc.pr.sum() + 1e-9)

    marl_recov_rate = np.mean(
        (R_m[CONTAIN_T:] / (R_m[CONTAIN_T:] + I_m[CONTAIN_T:] + 1e-9)) * pr_top_weight)
    sir_recov_rate  = np.mean(
        (R_s[CONTAIN_T:] / (R_s[CONTAIN_T:] + I_s[CONTAIN_T:] + 1e-9)) * pr_top_weight)

    IE = (marl_recov_rate - sir_recov_rate) / (sir_recov_rate + 1e-9)

    return {
        'IE':               float(IE),
        'marl_recov_rate':  float(marl_recov_rate),
        'sir_recov_rate':   float(sir_recov_rate),
        'pr_top_weight':    float(pr_top_weight),
        'top_k':            top_k,
        'hub_I_marl':       I_m * pr_top_weight,
        'hub_I_sir':        I_s * pr_top_weight,
    }


def plot_influential_efficiency(gc, hist_marl, hist_sir, fname=None):
    """Plot hub-weighted infected curves and bar chart of hub recovery rates with IE annotation."""
    ie = influential_efficiency(gc, hist_marl, hist_sir)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.plot(ie['hub_I_sir'],  'b--', linewidth=2, label='SIR hub-weighted I')
    ax.plot(ie['hub_I_marl'], 'r-',  linewidth=2, label='MARL hub-weighted I')
    ax.axvline(CONTAIN_T, color='gray', linestyle=':', label=f'Contain phase t={CONTAIN_T}')
    ax.set_title(f'Hub-Weighted Infection Curves\n'
                 f'(top {ie["top_k"]} nodes by PageRank, PR-weight={ie["pr_top_weight"]:.3f})')
    ax.set_xlabel('Time'); ax.set_ylabel('Hub-weighted infected')
    ax.legend(); ax.grid(alpha=0.3)

    ax2 = axes[1]
    labels = ['SIR (baseline)', 'MARL-DQN']
    rates  = [ie['sir_recov_rate'], ie['marl_recov_rate']]
    colors = ['#1976D2', '#D32F2F']
    bars   = ax2.bar(labels, rates, color=colors, edgecolor='black', alpha=0.85, width=0.4)
    for bar, val in zip(bars, rates):
        ax2.text(bar.get_x() + bar.get_width()/2, val + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
    ax2.set_title(f'Hub-Node Recovery Rate (Containment Phase)\n'
                  f'Influential Efficiency IE = {ie["IE"]:+.3f}')
    ax2.set_ylabel('PR-weighted recovery rate'); ax2.grid(axis='y', alpha=0.3)

    fig.suptitle('Influential Efficiency: MARL vs SIR', fontsize=12, fontweight='bold')
    fig.tight_layout(); plt.show()
    if fname: fig.savefig(f'outputs/{fname}.png', dpi=110, bbox_inches='tight')

    print(f'\n── Influential Efficiency ──────────────────────────────')
    print(f'  Top-{ie["top_k"]} hub nodes (top {100*ie["top_k"]/gc.n:.0f}% by PageRank)')
    print(f'  PR concentration weight : {ie["pr_top_weight"]:.4f}')
    print(f'  Hub recovery rate — SIR : {ie["sir_recov_rate"]:.4f}')
    print(f'  Hub recovery rate — MARL: {ie["marl_recov_rate"]:.4f}')
    print(f'  Influential Efficiency  : {ie["IE"]:+.3f}  (>0 = MARL leverages hubs better)')
    return ie


def simulate_multi_rumour(gc, marl_system=None, n_rumours=10,
                          detection_threshold=0.05, max_time=MAX_TIME, seed=SEED):
    """
    Simulate N independent rumour outbreaks on the same graph.

    Each outbreak starts from a fresh random seed set of INIT_INFECTED nodes.
    A rumour is flagged as 'detected' the first time the recovered fraction R(t)/n
    exceeds detection_threshold — enough systemic awareness to signal containment.

    marl_system=None runs the classical SIR baseline (no agent policy).

    Returns detection rates, detection timesteps, and per-rumour severity metrics.
    """
    rng = np.random.default_rng(seed)
    n   = gc.n
    results = {
        'n_generated':       n_rumours,
        'n_detected':        0,
        'detection_times':   [],
        'peak_I_list':       [],
        'final_R_list':      [],
        'containment_rates': [],
        'rumour_curves':     [],
    }

    for rumour_idx in range(n_rumours):
        states     = np.zeros(n, dtype=np.int64)
        init_nodes = rng.choice(n, INIT_INFECTED, replace=False)
        states[init_nodes] = 1
        h = torch.zeros(n, HIDDEN)

        if marl_system is not None:
            for net in marl_system.online:
                net.eval()

        I_curve = []; R_curve = []
        detected = False; detect_t = None

        with torch.no_grad():
            for t in range(max_time):
                if marl_system is not None:
                    x          = gc.features(states, t)
                    actions, h = marl_system.select_actions(x, h, eps=0.0)
                    states     = env_step(states, actions, gc, t, init_nodes=init_nodes)
                else:
                    # Classical SIR step with intensity modulation
                    intensity = rumour_intensity(t)
                    new = states.copy()
                    for u in np.where(states == 1)[0]:
                        for v in gc.nbrs[u]:
                            if states[v] == 0 and np.random.rand() < 0.2 * intensity:
                                new[v] = 1
                        if np.random.rand() < 0.05:
                            new[u] = 2
                    states = new

                n_I = int(np.sum(states == 1))
                n_R = int(np.sum(states == 2))
                I_curve.append(n_I); R_curve.append(n_R)

                # Detection event: first time recovered fraction crosses threshold
                if not detected and (n_R / n) >= detection_threshold:
                    detected = True; detect_t = t
                    results['n_detected'] += 1

        results['detection_times'].append(detect_t)
        results['peak_I_list'].append(int(max(I_curve)))
        results['final_R_list'].append(n_R)
        results['containment_rates'].append(n_R / n)
        results['rumour_curves'].append({'I': I_curve, 'R': R_curve})

    results['detection_rate'] = results['n_detected'] / n_rumours
    results['mean_detect_t']  = float(np.mean(
        [t for t in results['detection_times'] if t is not None] or [max_time]))
    results['mean_peak_I']    = float(np.mean(results['peak_I_list']))
    results['mean_final_R']   = float(np.mean(results['final_R_list']))
    return results


def plot_rumour_metrics(res_sir, res_marl, n_rumours, fname=None):
    """Six-panel comparison of rumour generation and detection metrics between SIR and MARL."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    ax = axes[0, 0]
    dr   = [res_sir['detection_rate'], res_marl['detection_rate']]
    bars = ax.bar(['SIR', 'MARL-DQN'], [v*100 for v in dr],
                  color=['#1976D2','#D32F2F'], edgecolor='black', alpha=0.85)
    for bar, val in zip(bars, dr):
        ax.text(bar.get_x()+bar.get_width()/2, val*100+0.5,
                f'{val*100:.0f}%', ha='center', fontweight='bold')
    ax.set_title(f'Detection Rate ({n_rumours} rumours generated each)')
    ax.set_ylabel('% Rumours Detected'); ax.set_ylim(0, 110); ax.grid(axis='y', alpha=0.3)

    ax = axes[0, 1]
    dt   = [res_sir['mean_detect_t'], res_marl['mean_detect_t']]
    bars = ax.bar(['SIR', 'MARL-DQN'], dt,
                  color=['#1976D2','#D32F2F'], edgecolor='black', alpha=0.85)
    for bar, val in zip(bars, dt):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.2, f't={val:.1f}', ha='center', fontweight='bold')
    ax.set_title('Mean Detection Timestep (lower = faster)'); ax.set_ylabel('Timestep'); ax.grid(axis='y', alpha=0.3)

    ax = axes[0, 2]
    pi   = [res_sir['mean_peak_I'], res_marl['mean_peak_I']]
    bars = ax.bar(['SIR', 'MARL-DQN'], pi,
                  color=['#1976D2','#D32F2F'], edgecolor='black', alpha=0.85)
    for bar, val in zip(bars, pi):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.2, f'{val:.0f}', ha='center', fontweight='bold')
    ax.set_title('Mean Peak Infected per Rumour'); ax.set_ylabel('Peak infected nodes'); ax.grid(axis='y', alpha=0.3)

    ax = axes[1, 0]
    ax.scatter(range(n_rumours), res_sir['peak_I_list'],  color='#1976D2', alpha=0.7, label='SIR',  s=50)
    ax.scatter(range(n_rumours), res_marl['peak_I_list'], color='#D32F2F', alpha=0.7, label='MARL', s=50, marker='^')
    ax.axhline(res_sir['mean_peak_I'],  color='#1976D2', linestyle='--', alpha=0.6)
    ax.axhline(res_marl['mean_peak_I'], color='#D32F2F', linestyle='--', alpha=0.6)
    ax.set_title('Peak Infected per Rumour'); ax.set_xlabel('Rumour index')
    ax.set_ylabel('Peak I'); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1, 1]
    sir_dt  = [t if t is not None else MAX_TIME for t in res_sir['detection_times']]
    marl_dt = [t if t is not None else MAX_TIME for t in res_marl['detection_times']]
    ax.scatter(range(n_rumours), sir_dt,  color='#1976D2', alpha=0.7, label='SIR',  s=50)
    ax.scatter(range(n_rumours), marl_dt, color='#D32F2F', alpha=0.7, label='MARL', s=50, marker='^')
    ax.axhline(MAX_TIME, color='gray', linestyle=':', label=f'Not detected (t={MAX_TIME})')
    ax.set_title('Detection Timestep per Rumour'); ax.set_xlabel('Rumour index')
    ax.set_ylabel('Detection t'); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1, 2]
    for c in res_sir['rumour_curves']:
        ax.plot(c['I'], color='#1976D2', alpha=0.25, linewidth=1)
    for c in res_marl['rumour_curves']:
        ax.plot(c['I'], color='#D32F2F', alpha=0.25, linewidth=1)
    ax.plot([], color='#1976D2', label='SIR runs')
    ax.plot([], color='#D32F2F', label='MARL-DQN runs')
    ax.set_title(f'All {n_rumours} Rumour I-Curves Overlay')
    ax.set_xlabel('Time'); ax.set_ylabel('Infected'); ax.legend(); ax.grid(alpha=0.3)

    fig.suptitle(f'Rumour Generation & Detection Analysis — {n_rumours} Rumours | '
                 f'SIR detected {res_sir["n_detected"]} | MARL detected {res_marl["n_detected"]}',
                 fontsize=12, fontweight='bold')
    fig.tight_layout(); plt.show()
    if fname: fig.savefig(f'outputs/{fname}.png', dpi=110, bbox_inches='tight')


def print_rumour_table(res_sir, res_marl):
    print(f'\n── Rumour Generation & Detection Summary ──────────────────────────')
    print(f'  {"Metric":<35} {"SIR":>10} {"MARL-DQN":>12}')
    print(f'  {"-"*60}')
    print(f'  {"Rumours generated":<35} {res_sir["n_generated"]:>10} {res_marl["n_generated"]:>12}')
    print(f'  {"Rumours detected":<35} {res_sir["n_detected"]:>10} {res_marl["n_detected"]:>12}')
    print(f'  {"Detection rate":<35} {res_sir["detection_rate"]:>10.1%} {res_marl["detection_rate"]:>12.1%}')
    print(f'  {"Mean detection timestep":<35} {res_sir["mean_detect_t"]:>10.1f} {res_marl["mean_detect_t"]:>12.1f}')
    print(f'  {"Mean peak infected":<35} {res_sir["mean_peak_I"]:>10.1f} {res_marl["mean_peak_I"]:>12.1f}')
    print(f'  {"Mean final recovered":<35} {res_sir["mean_final_R"]:>10.1f} {res_marl["mean_final_R"]:>12.1f}')
    improvement = (res_sir['mean_peak_I'] - res_marl['mean_peak_I']) / (res_sir['mean_peak_I'] + 1e-9)
    faster      = res_sir['mean_detect_t'] - res_marl['mean_detect_t']
    print(f'  {"Peak-I reduction (MARL vs SIR)":<35} {improvement:>22.1%}')
    print(f'  {"MARL detects faster by (steps)":<35} {faster:>22.1f}')

print('Plotting helpers + influential efficiency + rumour metrics defined.')


## Section 1 — Train MARL & Compare with Classical SIR

In [ ]:
marl_ba, curves = train_marl(gtype='ba', verbose=True)
plot_training_curves(curves, fname='01_training_curves')

h_sir  = run_sir()
h_marl = eval_marl(marl_ba)
plot_sir_pair(h_sir, h_marl, fname='02_sir_vs_marl_ba')
print_table('SIR vs Proper MARL (BA)', [('Classical SIR', h_sir), ('Proper MARL-DQN', h_marl)])

# ── Overlay I-curves: MARL vs SIR — the key divergence plot ─────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(h_sir['I'],  'r--', linewidth=2, label='Classical SIR (fixed β=0.2)')
ax.plot(h_marl['I'], 'r-',  linewidth=2.5, label='MARL-DQN (learned policy)')
ax.fill_between(range(MAX_TIME),
                h_sir['I'], h_marl['I'],
                where=[h_marl['I'][t] < h_sir['I'][t] for t in range(MAX_TIME)],
                alpha=0.2, color='green', label='MARL containment advantage')
ax.axvline(CONTAIN_T, color='black', linestyle='--', linewidth=1.5,
           label=f'Containment phase starts (t={CONTAIN_T})')
ax.set_title('Infected Over Time: Classical SIR vs MARL-DQN\n'
             'MARL grows faster (ignition window) then contains faster (learned refutation)',
             fontsize=11)
ax.set_xlabel('Time'); ax.set_ylabel('Infected nodes')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()
fig.savefig('outputs/02b_I_comparison.png', dpi=110, bbox_inches='tight')

# Print the key numbers for the paper
i_sir  = np.array(h_sir['I'])
i_marl = np.array(h_marl['I'])
print(f"\n{'='*50}")
print(f"SIR  peak: {max(i_sir):3d} at t={np.argmax(i_sir)}")
print(f"MARL peak: {max(i_marl):3d} at t={np.argmax(i_marl)}")
contain_sir  = int(np.mean(i_sir[CONTAIN_T:]))
contain_marl = int(np.mean(i_marl[CONTAIN_T:]))
print(f"\nMean I after t={CONTAIN_T} (containment phase):")
print(f"  SIR : {contain_sir} avg infected")
print(f"  MARL: {contain_marl} avg infected")
if contain_sir > 0:
    reduction = (contain_sir - contain_marl) / contain_sir * 100
    print(f"  MARL reduces sustained infection by {reduction:.1f}%")
print(f"{'='*50}")


## Section 2 — Twitter Dataset (Real-World Network)

Applies the trained MARL policy to a real social network from the Stanford SNAP Twitter dataset.
*(Requires `twitter_combined.txt` in the working directory — download from https://snap.stanford.edu/data/ego-Twitter.html)*


**Note (revision, November 2026): the next cell is superseded.**

It assigns communities with `marl_ba.node_community[:gc_tw.n]`, but `node_community` has only 5,000 entries (the BA training graph). Only the first 5,000 Twitter users therefore act on the policy; the other 76,306 default to Ignore at every step. The Twitter results in the revised paper come from `run_twitter_transfer_fixed.py`, which detects communities on the Twitter graph itself so that all 81,306 users act on the policy.

In [ ]:
# Apply trained MARL policy to the Twitter ego-network dataset (Stanford SNAP)

if os.path.exists("twitter_combined.txt"):
    # Load and relabel the Twitter network to contiguous integer node indices
    G_tw = nx.convert_node_labels_to_integers(
        nx.read_edgelist("twitter_combined.txt", create_using=nx.Graph()))
    print(f"Twitter graph: {G_tw.number_of_nodes()} nodes, {G_tw.number_of_edges()} edges")

    gc_tw = GraphCache(G_tw)

    # Initialise simulation state on the Twitter graph
    states = np.zeros(gc_tw.n, dtype=np.int64)
    init_nodes_tw = np.random.choice(gc_tw.n, min(INIT_INFECTED, gc_tw.n // 10), replace=False)
    states[init_nodes_tw] = 1

    h_tw  = torch.zeros(gc_tw.n, HIDDEN)
    hist_tw = {"S": [], "I": [], "R": []}

    # Override the MARL system's graph cache to the Twitter graph for decentralised execution
    marl_ba.gc = gc_tw

    marl_ba.community_masks = [
        np.where(marl_ba.node_community[:gc_tw.n] == c)[0] for c in range(N_COMMUNITIES)
    ]

    for net in marl_ba.online:
        net.eval()

    with torch.no_grad():
        for t in range(MAX_TIME):
            x_tw       = gc_tw.features(states, t)
            acts_tw, h_tw = marl_ba.select_actions(x_tw, h_tw, eps=0.0)
            states     = env_step(states, acts_tw, gc_tw, t, init_nodes=init_nodes_tw)
            hist_tw["S"].append(int(np.sum(states == 0)))
            hist_tw["I"].append(int(np.sum(states == 1)))
            hist_tw["R"].append(int(np.sum(states == 2)))

    # Plot SIR dynamics on the Twitter network
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(hist_tw["S"], "b-", label="Susceptible")
    ax.plot(hist_tw["I"], "r-", label="Infected (spreading)")
    ax.plot(hist_tw["R"], "g-", label="Recovered (refuted)")
    ax.axvline(CONTAIN_T, color='black', linestyle='--', linewidth=1.5,
               label=f'Containment phase (t={CONTAIN_T})')
    ax.set_title("MARL-DQN Applied to Twitter Ego-Network (Stanford SNAP)")
    ax.set_xlabel("Time"); ax.set_ylabel("Nodes")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    fig.savefig("outputs/03_twitter_marl.png", dpi=110, bbox_inches="tight")

    print_table("Twitter Network — MARL-DQN", [("Twitter MARL", hist_tw)])
else:
    print("twitter_combined.txt not found.")


## Section 3 — Ablation: Component Removal

| Condition | What changes |
|---|---|
| Full MARL-DQN | trained model, shaped reward, three node types, phase-split reward |
| No Social Contagion | passive SIR only, no Share/Refute actions |
| No MARL (random policy) | untrained model |
| No EGT Utility Shaping | flat ±1 reward, no node types, no phase-split |
| No Ignition Window | seed nodes treated like all others (no forced Share) |


In [ ]:
# ── Ablation Condition 1: Full trained MARL (already trained above) ─────────
h_full = eval_marl(marl_ba)

# ── Ablation Condition 2: No Social Contagion — passive SIR without agent actions ─
def passive_sir(max_time=MAX_TIME):
    """
    Passive SIR baseline with no agent actions — spread occurs only via BASE_SPREAD.
    Isolates the contribution of active Share/Refute decisions to rumour dynamics.
    """
    G = make_graph('ba'); n = G.number_of_nodes()
    s = np.zeros(n, dtype=np.int64)
    s[np.random.choice(n, INIT_INFECTED, replace=False)] = 1
    hist = {'S': [], 'I': [], 'R': []}
    for t in range(max_time):
        intensity = rumour_intensity(t)
        new = s.copy()
        for u in G.nodes():
            if s[u] == 1:
                for v in G.neighbors(u):
                    if s[v] == 0 and np.random.rand() < BASE_SPREAD * intensity:
                        new[v] = 1
            if s[u] == 1 and np.random.rand() < 0.03:
                new[u] = 2
        s = new
        hist['S'].append(int(np.sum(s == 0)))
        hist['I'].append(int(np.sum(s == 1)))
        hist['R'].append(int(np.sum(s == 2)))
    return hist

h_nosc = passive_sir()

# ── Ablation Condition 3: No MARL — random untrained policy ──────────────
gc_rand   = GraphCache(make_graph('ba'))
marl_rand = MARLSystem(gc_rand)
h_rand    = eval_marl(marl_rand)

# ── Ablation Condition 4: No EGT — retrain with flat ±1 reward signal ────
marl_noegt, _ = train_marl(gtype='ba', flat_reward=True, label='No-EGT', verbose=True)
h_noegt = eval_marl(marl_noegt)

# ── Ablation Condition 5: No Ignition Window — seed nodes not forced to Share ─
def train_marl_no_ignition(gtype='ba', n_eps=TRAIN_EPS):
    """
    Trains without the superspreader ignition window (init_nodes=None in env_step).
    Isolates the contribution of Fix 2 (forced Share for seed nodes during t < PEAK_T).
    """
    gc_ni   = GraphCache(make_graph(gtype))
    marl_ni = MARLSystem(gc_ni)
    eps_ni  = EPS_START
    SUBSAMPLE = max(1, gc_ni.n // 50)

    for ep in range(n_eps):
        states_ni  = np.zeros(gc_ni.n, dtype=np.int64)
        states_ni[np.random.choice(gc_ni.n, INIT_INFECTED, replace=False)] = 1
        h_ni = torch.zeros(gc_ni.n, HIDDEN)

        for t in range(MAX_TIME):
            x_ni              = gc_ni.features(states_ni, t)
            actions_ni, h_ni_new = marl_ni.select_actions(x_ni, h_ni, eps_ni)
            new_states_ni     = env_step(states_ni, actions_ni, gc_ni, t, init_nodes=None)
            rew_ni            = reward_fn(states_ni, actions_ni, gc_ni, t, flat=False)
            x_next_ni         = gc_ni.features(new_states_ni, t + 1)
            done_ni           = float(t == MAX_TIME - 1)

            for ii in range(0, gc_ni.n, SUBSAMPLE):
                c_ni = marl_ni.node_community[ii]
                marl_ni.buffers[c_ni].push(
                    x_ni[ii].numpy(), actions_ni[ii], rew_ni[ii],
                    x_next_ni[ii].numpy(), done_ni, node_idx=ii)

            for c_ni in range(N_COMMUNITIES):
                marl_ni.update(c_ni)

            states_ni = new_states_ni
            h_ni      = h_ni_new.detach()

        if (ep + 1) % TARGET_UPDATE == 0:
            marl_ni.sync_targets()
        eps_ni = max(EPS_END, eps_ni * EPS_DECAY)

    return marl_ni

marl_noign = train_marl_no_ignition()
h_noign    = eval_marl(marl_noign)
print('No-Ignition-Window ablation trained.')

abl = [
    ('Full Proper MARL-DQN',   h_full),
    ('No Social Contagion',    h_nosc),
    ('No MARL (random policy)',h_rand),
    ('No EGT Utility Shaping', h_noegt),
    ('No Ignition Window',     h_noign),
]

# Plot SIR curves for all ablation conditions
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, (lbl, h) in zip(axes.flatten(), abl):
    ax.plot(h['S'], 'b-', label='S')
    ax.plot(h['I'], 'r-', label='I')
    ax.plot(h['R'], 'g-', label='R')
    ax.set_title(lbl); ax.set_xlabel('Time'); ax.set_ylabel('Nodes')
    ax.legend(); ax.grid(alpha=0.3)
for ax in axes.flatten()[len(abl):]:
    ax.set_visible(False)
fig.suptitle('Ablation: Component Removal', fontsize=12, fontweight='bold')
fig.tight_layout(); plt.show()
fig.savefig('outputs/04_ablation_sir.png', dpi=110, bbox_inches='tight')

# Plot polarization index for all ablation conditions
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, (lbl, h) in zip(axes.flatten(), abl):
    ax.plot(glob_pol(h), color='purple', label='Polarization')
    ax.set_title(f'{lbl} — Polarization')
    ax.set_xlabel('Time'); ax.set_ylabel('Index')
    ax.legend(); ax.grid(alpha=0.3)
for ax in axes.flatten()[len(abl):]:
    ax.set_visible(False)
fig.suptitle('Ablation: Polarization Index', fontsize=12, fontweight='bold')
fig.tight_layout(); plt.show()
fig.savefig('outputs/05_ablation_polarization.png', dpi=110, bbox_inches='tight')

print_table('Ablation — Component Removal', abl)


## Section 4 — Ablation: Initial Infected Nodes

In [ ]:
sir_init=[]; marl_init=[]
for k in [5, 25, 50, 100]:
    sir_init.append((f'Init={k}', run_sir(init_infected=k)))
    m, _ = train_marl(gtype='ba', init_infected=k, verbose=False)
    marl_init.append((f'Init={k}', eval_marl(m, init_infected=k)))

plot_multi_I(sir_init,  'SIR: Effect of Initial Infected Nodes',  fname='06_init_sir')
plot_multi_I(marl_init, 'MARL: Effect of Initial Infected Nodes', fname='07_init_marl')
print_table('Initial Infected — SIR',  sir_init)
print_table('Initial Infected — MARL', marl_init)


## Section 5 — Ablation: Parameter Sweeps (β, γ, α, β_c, γ_b, node_type_mix)


In [ ]:
sir_beta = [(f'β={b}', run_sir(beta=b, gamma=0.05, init_infected=50)) for b in [0.05,0.1,0.2,0.3]]
plot_multi_I(sir_beta, 'SIR: Effect of Beta', fname='08_sir_beta')
print_table('SIR Beta Sweep', sir_beta)

sir_gam = [(f'γ={g}', run_sir(beta=0.2, gamma=g, init_infected=50)) for g in [0.02,0.05,0.1,0.2]]
plot_multi_I(sir_gam, 'SIR: Effect of Gamma', fname='09_sir_gamma')
print_table('SIR Gamma Sweep', sir_gam)

marl_alpha = []
for a in [0.3, 0.6, 0.9]:
    m, _ = train_marl(gtype='ba', verbose=False, label=f'α={a}')
    marl_alpha.append((f'α={a}', eval_marl(m)))
plot_multi_I(marl_alpha, 'MARL: Effect of Alpha (Peer Pressure Weight)', fname='10_marl_alpha')
print_table('MARL Alpha Sweep', marl_alpha)

marl_bc = []
for bc in [0.1, 0.2, 0.4]:
    m, _ = train_marl(gtype='ba', verbose=False, label=f'β_c={bc}')
    marl_bc.append((f'β_c={bc}', eval_marl(m)))
plot_multi_I(marl_bc, 'MARL: Effect of Beta_c (Cost Weight)', fname='11_marl_betac')
print_table('MARL Beta_c Sweep', marl_bc)

marl_gb = []
for gb in [0.2, 0.4, 0.8]:
    m, _ = train_marl(gtype='ba', verbose=False, label=f'γ_b={gb}')
    marl_gb.append((f'γ_b={gb}', eval_marl(m)))
plot_multi_I(marl_gb, 'MARL: Effect of Gamma_b (Fatigue Rate)', fname='12_marl_gammab')
print_table('MARL Gamma_b Sweep', marl_gb)

marl_ntmix = []
for mix_label, probs in [
    ('Mostly Informed (50/30/20)',   [0.50, 0.30, 0.20]),
    ('Mostly Uninformed (10/70/20)', [0.10, 0.70, 0.20]),
    ('Mostly Indifferent (20/30/50)',[0.20, 0.30, 0.50]),
]:
    # Temporarily override NODE_TYPE_PROBS by patching GraphCache.assign
    import builtins
    orig_probs = NODE_TYPE_PROBS[:]
    NODE_TYPE_PROBS[:] = probs  # mutate in-place (it's a list)
    m, _ = train_marl(gtype='ba', verbose=False, label=mix_label)
    marl_ntmix.append((mix_label, eval_marl(m)))
    NODE_TYPE_PROBS[:] = orig_probs  # restore

plot_multi_I(marl_ntmix, 'MARL: Effect of Node-Type Mix (Informed/Uninformed/Indifferent)',
             fname='18_marl_nodetypemix')
print_table('MARL Node-Type Mix Sweep', marl_ntmix)


## Section 6 — Ablation: Time Horizon & Utility Decay

In [ ]:
sir_T  = [(f'T={T}', run_sir(init_infected=50, max_time=T)) for T in [40, 80, 120]]
plot_multi_I(sir_T, 'SIR: Effect of Time Horizon', fname='13_sir_time')
print_table('SIR Time Horizon', sir_T)

marl_T = []
for T in [40, 80, 120]:
    m, _ = train_marl(gtype='ba', verbose=False, label=f'T={T}')
    marl_T.append((f'T={T}', eval_marl(m, max_time=T)))
plot_multi_I(marl_T, 'MARL: Effect of Time Horizon', fname='14_marl_time')
print_table('MARL Time Horizon', marl_T)

marl_dec = []
for d in [0.01, 0.02, 0.05]:
    m, _ = train_marl(gtype='ba', verbose=False, label=f'fatigue={d}')
    marl_dec.append((f'fatigue={d}', eval_marl(m)))
plot_multi_I(marl_dec, 'MARL: Rumour Fatigue Rate Sweep', fname='15_marl_decay')
print_table('MARL Fatigue Sweep', marl_dec)


## Section 7 — Ablation: Network Topology

In [ ]:
topo_sir=[]; topo_marl=[]
for gt, lbl in [('ba','Barabasi-Albert'),('er','Erdos-Renyi'),
                ('ws','Watts-Strogatz'),('sbm','Stoch. Block Model')]:
    m, _ = train_marl(gtype=gt, verbose=True, label=lbl)
    hs   = run_sir(gtype=gt)
    hm   = eval_marl(m)
    plot_sir_pair(hs, hm, title=f' ({lbl})', fname=f'16_topo_{gt}')
    topo_sir.append((lbl, hs)); topo_marl.append((lbl, hm))

print_table('Topology — SIR',  topo_sir)
print_table('Topology — MARL', topo_marl)


## Section 7b — Ablation: PageRank-weighted growth-phase reward

We explore whether explicitly rewarding spread through influential nodes (PageRank-weighted growth-phase reward) produces different containment dynamics. This addresses whether TEMPEST's effectiveness depends on hub-targeting behavior or generalizes across spreading strategies.

**Method:** The hub-weighted reward inflates the growth-phase Share reward for high-PageRank nodes: `r_S = 0.15 × (1 + PR_i × n × 0.5)` and `r_I = 0.4 × intensity × (1 + PR_i × n × 0.5)`. All other reward terms are unchanged. We train on a 2 000-node BA graph for 40 episodes (qualitative ablation only — headline results use 5 000 nodes / 80 episodes).

In [ ]:
import os, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
os.makedirs('outputs', exist_ok=True)

# ── Ablation settings ────────────────────────────────────────────────────
ABL_NODES = 5000
ABL_EPS   = 80

# ── Hub-weighted reward function ─────────────────────────────────────────
def reward_fn_hub_weighted(states, actions, gc, t, flat=False):
    """
    Identical to reward_fn but growth-phase Share rewards scale with PageRank.
    Susceptible + Share + growth: r = 0.15 * (1 + gc.pr[i] * gc.n * 0.5)
    Infected    + Share + growth: r = 0.4 * intensity * (1 + gc.pr[i] * gc.n * 0.5)
    """
    intensity  = rumour_intensity(t)
    r          = np.zeros(gc.n, np.float32)
    node_types = gc.node_types
    in_contain = (t >= CONTAIN_T)

    if flat:
        for i in range(gc.n):
            if   states[i] == 1 and actions[i] == 2: r[i] =  1.0
            elif states[i] == 0 and actions[i] == 1: r[i] = -1.0
        return r

    pr_ratio = np.array(
        [sum(states[j]==1 for j in gc.nbrs[i]) / (len(gc.nbrs[i])+1)
         for i in range(gc.n)], np.float32)
    egt = egt_payoff(node_types, actions, pr_ratio, intensity)

    for i in range(gc.n):
        a = actions[i]
        if states[i] == 0:      # Susceptible
            if not in_contain:
                r[i] = 0.15 * (1.0 + gc.pr[i] * gc.n * 0.5) if a == 1 else 0.0
            else:
                if   a == 1: r[i] = -2.0 * intensity
                elif a == 2: r[i] =  0.5 + 0.4 * (1.0 - pr_ratio[i])
                else:        r[i] =  0.1
        elif states[i] == 1:    # Infected
            if not in_contain:
                if   a == 1: r[i] = 0.4 * intensity * (1.0 + gc.pr[i] * gc.n * 0.5)
                elif a == 2: r[i] = -0.1
                else:        r[i] =  0.0
            else:
                if   a == 2: r[i] =  4.0
                elif a == 1: r[i] = -2.0 * intensity * pr_ratio[i]
                else:        r[i] = -0.3
        else:                   # Recovered
            r[i] = 0.3 if (a == 2 and in_contain) else 0.05
        r[i] *= (1.0 + 0.5 * egt[i])
    return r


# ── Hub-aware eval (tracks top-PR nodes separately) ──────────────────────
def eval_marl_hub(marl_system, top_frac=0.10):
    """Greedy rollout; also tracks I/R among top-PageRank nodes."""
    gc     = marl_system.gc
    n_top  = max(1, int(gc.n * top_frac))
    top_set= set(np.argsort(gc.pr)[-n_top:])
    states = np.zeros(gc.n, dtype=np.int64)
    init_n = np.random.choice(gc.n, INIT_INFECTED, replace=False)
    states[init_n] = 1
    import torch
    h = torch.zeros(gc.n, HIDDEN)
    hist = {'S':[], 'I':[], 'R':[], 'hub_I':[], 'hub_R':[]}
    for net in marl_system.online: net.eval()
    with torch.no_grad():
        for t in range(MAX_TIME):
            x = gc.features(states, t)
            actions, h = marl_system.select_actions(x, h, eps=0.0)
            states = env_step(states, actions, gc, t, init_nodes=init_n)
            hist['S'].append(int(np.sum(states==0)))
            hist['I'].append(int(np.sum(states==1)))
            hist['R'].append(int(np.sum(states==2)))
            hist['hub_I'].append(sum(1 for i in top_set if states[i]==1))
            hist['hub_R'].append(sum(1 for i in top_set if states[i]==2))
    return hist


def hub_recov_rate(hist):
    """PR-weighted recovery rate in containment phase (mirrors influential_efficiency metric)."""
    I = np.array(hist['I']); R = np.array(hist['R'])
    return float(np.mean(R[CONTAIN_T:] / (R[CONTAIN_T:] + I[CONTAIN_T:] + 1e-9)))


def sustained_red(hist_sir, hist_marl):
    I_s = np.array(hist_sir['I'])[CONTAIN_T:].mean()
    I_m = np.array(hist_marl['I'])[CONTAIN_T:].mean()
    return (I_s - I_m) / (I_s + 1e-9) * 100


# ── Monkey-patch make_graph to return 2 000-node graph for both trainings ─
_orig_make_graph = make_graph
_orig_reward_fn  = reward_fn

def _abl_make_graph(gtype='ba', n=ABL_NODES):
    return _orig_make_graph(gtype, n=ABL_NODES)

make_graph = _abl_make_graph

print(f'Training standard MARL ({ABL_NODES} nodes, {ABL_EPS} eps)...')
marl_std_abl, _ = train_marl(n_eps=ABL_EPS, verbose=True, label='Std reward')

print(f'Training hub-weighted MARL ({ABL_NODES} nodes, {ABL_EPS} eps)...')
reward_fn = reward_fn_hub_weighted
marl_hub_abl, _ = train_marl(n_eps=ABL_EPS, verbose=True, label='Hub-weighted')

# Restore globals
reward_fn  = _orig_reward_fn
make_graph = _orig_make_graph

# ── Run SIR baseline on same 2 000-node graph ────────────────────────────
def run_sir_gc(gc_in, beta=0.2, gamma=0.02, init_infected=INIT_INFECTED, max_time=MAX_TIME):
    """SIR baseline on an existing GraphCache (no new graph created)."""
    G = gc_in.G; n = gc_in.n
    states = np.zeros(n, dtype=np.int64)
    states[np.random.choice(n, init_infected, replace=False)] = 1
    hist = {'S': [], 'I': [], 'R': []}
    for t in range(max_time):
        intensity = rumour_intensity(t)
        new = states.copy()
        for u in np.where(states == 1)[0]:
            for v in G.neighbors(u):
                if states[v] == 0 and np.random.rand() < beta * intensity:
                    new[v] = 1
            if np.random.rand() < gamma:
                new[u] = 2
        states = new
        hist['S'].append(int(np.sum(states==0)))
        hist['I'].append(int(np.sum(states==1)))
        hist['R'].append(int(np.sum(states==2)))
    return hist

h_sir_abl  = run_sir_gc(marl_std_abl.gc)
h_std_abl  = eval_marl_hub(marl_std_abl)
h_hub_abl  = eval_marl_hub(marl_hub_abl)

# ── Comparison table ─────────────────────────────────────────────────────
rows = [
    ('Standard MARL', h_std_abl),
    ('Hub-weighted MARL', h_hub_abl),
]
print(f'\n{"Variant":<22} {"Peak I":>8} {"Final R":>9} {"Sust. red %":>12} {"Hub recov"+" rate":>12}')
print('-' * 68)
for name, h in rows:
    peak  = max(h['I'])
    finalR= h['R'][-1]
    sred  = sustained_red(h_sir_abl, h)
    hrate = hub_recov_rate(h)
    print(f'{name:<22} {peak:>8} {finalR:>9} {sred:>11.1f}% {hrate:>12.3f}')

# ── Side-by-side I-curves ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
T = range(MAX_TIME)
for ax, h, title in zip(axes,
                         [h_std_abl, h_hub_abl],
                         ['Standard reward', 'Hub-weighted reward']):
    ax.plot(T, h_sir_abl['I'], 'r-', lw=2, alpha=0.7, label='SIR baseline')
    ax.plot(T, h['I'],          'b-', lw=2, label='MARL-DQN')
    ax.fill_between(T,
                    [min(s, m) for s,m in zip(h_sir_abl['I'], h['I'])],
                    [max(s, m) for s,m in zip(h_sir_abl['I'], h['I'])],
                    alpha=0.15, color='green', label='Gap')
    ax.axvline(CONTAIN_T, color='black', ls='--', lw=1.2,
               label=f'Contain (t={CONTAIN_T})')
    ax.set_title(f'MARL vs SIR — {title}', fontsize=11)
    ax.set_xlabel('Simulation step'); ax.set_ylabel('Infected nodes')
    ax.legend(fontsize=9); ax.grid(alpha=0.25)

fig.suptitle(
    f'Hub-Targeting Ablation ({ABL_NODES}-node BA graph, {ABL_EPS} training eps)',
    fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig('outputs/22_hub_targeting_ablation.png', dpi=130, bbox_inches='tight')
plt.close()
print('\nSaved → outputs/22_hub_targeting_ablation.png')


## Section 7c — Ablation: CONTAIN_T window sensitivity (post-hoc analysis)

TEMPEST was trained with `CONTAIN_T=30` as a fixed hyperparameter. Here we examine the sensitivity of our headline metric to this choice — specifically, whether the 79.1% sustained reduction holds when the analysis window is shifted. This is a post-hoc reanalysis of existing simulation curves; no policies were retrained.


In [ ]:
import json, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
os.makedirs('outputs', exist_ok=True)

# ── Load 20-seed I-curves ─────────────────────────────────────────────────
_seed_files = sorted(Path('multiseed_results').glob('seed_*.json'))
assert len(_seed_files) == 20, f'Expected 20 seeds, found {len(_seed_files)}'
_seed_data  = [json.load(open(sf)) for sf in _seed_files]
print(f'Loaded {len(_seed_data)} seeds')

# ── Compute sustained reduction for each CONTAIN_T candidate ─────────────
CONTAIN_T_CANDIDATES = [20, 25, 30, 35, 40, 45, 50]
HEADLINE_VAL = 79.1   # reference 79.1% from the 20-seed aggregate

ct_means, ct_sems = [], []
print(f'\n{"CONTAIN_T":>10} | {"mean ± SEM (n=20)":<25} | range')
print('-' * 60)

for ct in CONTAIN_T_CANDIDATES:
    reds = []
    for r in _seed_data:
        I_sir  = np.array(r['I_curve_sir'])[ct:]
        I_marl = np.array(r['I_curve_marl'])[ct:]
        if len(I_sir) == 0:
            reds.append(0.0)
            continue
        s_sir = I_sir.mean()
        pct   = (s_sir - I_marl.mean()) / (s_sir + 1e-9) * 100
        reds.append(pct)
    reds = np.array(reds)
    m    = reds.mean()
    sem  = reds.std(ddof=1) / np.sqrt(len(reds))
    ct_means.append(m)
    ct_sems.append(sem)
    flag = '  ← headline' if ct == 30 else ''
    print(f'{ct:>10} | {m:.1f}% ± {sem:.1f}%{flag:<20} | [{reds.min():.1f}%, {reds.max():.1f}%]')

# ── Plot ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.errorbar(CONTAIN_T_CANDIDATES, ct_means, yerr=ct_sems,
            fmt='o-', color='#1f77b4', lw=2, capsize=5,
            label='Sustained I reduction (mean ± SEM)')

ax.axhline(HEADLINE_VAL, color='#d62728', ls='--', lw=1.5,
           label=f'Headline value (79.1%, CONTAIN_T=30)')

# Shade SEM band around headline
headline_sem = 3.2
ax.axhspan(HEADLINE_VAL - headline_sem, HEADLINE_VAL + headline_sem,
           alpha=0.08, color='#d62728')

ax.axvline(30, color='black', ls=':', lw=1.2, alpha=0.6,
           label='Trained CONTAIN_T = 30')

ax.set_xlabel('CONTAIN_T (analysis window start)', fontsize=12)
ax.set_ylabel('Sustained infection reduction (%)', fontsize=12)
ax.set_title('CONTAIN_T Window Sensitivity (post-hoc, 20 seeds, no retraining)',
             fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(0, 105)
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig('outputs/23_containt_sensitivity.png', dpi=130, bbox_inches='tight')
plt.close()
print('\nSaved → outputs/23_containt_sensitivity.png')



## Section 8 — Full Metric Analysis (SIR vs MARL)

In [ ]:
I_s = np.array(h_sir['I']);  R_s = np.array(h_sir['R'])
I_m = np.array(h_marl['I']); R_m = np.array(h_marl['R'])
t_cut = 30

print(f'Spread Reduction      : {(np.max(I_s)-np.max(I_m))/(np.max(I_s)+1e-9):.3f}')
print(f'Peak Infected SIR     : {np.max(I_s)}')
print(f'Peak Infected MARL    : {np.max(I_m)}')
print(f'Propagation Idx SIR   : {np.mean(I_s):.1f}')
print(f'Propagation Idx MARL  : {np.mean(I_m):.1f}')
print(f'Truth Amplif. SIR     : {np.mean(R_s[:t_cut])/(np.mean(I_s[:t_cut])+1e-9):.3f}')
print(f'Truth Amplif. MARL    : {np.mean(R_m[:t_cut])/(np.mean(I_m[:t_cut])+1e-9):.3f}')
print(f'Global Pol. SIR       : {np.mean(np.abs(I_s-R_s)/(I_s+R_s+1e-9)):.3f}')
print(f'Global Pol. MARL      : {np.mean(np.abs(I_m-R_m)/(I_m+R_m+1e-9)):.3f}')

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(glob_pol(h_sir),  'b--', label='SIR Global Polarization')
ax.plot(glob_pol(h_marl), 'r--', label='MARL Global Polarization')
ax.set_title('Global Polarization: SIR vs Proper MARL-DQN')
ax.set_xlabel('Time'); ax.set_ylabel('Polarization Index'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
fig.savefig('outputs/17_polarization_comparison.png', dpi=110, bbox_inches='tight')


## Section 9 — Node-Type Breakdown Analysis

Breaks down the SIR curves by node type to show:
- **Indifferent** nodes create the plateau (slow mid-outbreak phase)
- **Informed** nodes drive the refutation curve
- **Uninformed** nodes are the primary infection vector


In [ ]:
# ── Section 9: Per-node-type SIR breakdown ───────────────────────
def eval_marl_typed(marl_system, init_infected=INIT_INFECTED, max_time=MAX_TIME):
    """
    Like eval_marl but tracks infection curves separately per node type.
    Returns hist_full + hist_by_type dict.
    """
    gc = marl_system.gc
    nt = gc.node_types
    states = np.zeros(gc.n, dtype=np.int64)
    init_nodes = np.random.choice(gc.n, init_infected, replace=False)
    states[init_nodes] = 1
    h = torch.zeros(gc.n, HIDDEN)
    type_names = ['Informed', 'Uninformed', 'Indifferent']
    hist_by_type = {name: {'S':[], 'I':[], 'R':[]} for name in type_names}
    hist_full = {'S':[], 'I':[], 'R':[]}
    for net in marl_system.online: net.eval()
    with torch.no_grad():
        for t in range(max_time):
            x = gc.features(states, t)
            actions, h = marl_system.select_actions(x, h, eps=0.0)
            states = env_step(states, actions, gc, t, init_nodes=init_nodes)
            hist_full['S'].append(int(np.sum(states==0)))
            hist_full['I'].append(int(np.sum(states==1)))
            hist_full['R'].append(int(np.sum(states==2)))
            for ti, name in enumerate(type_names):
                mask = (nt == ti)
                hist_by_type[name]['S'].append(int(np.sum(states[mask]==0)))
                hist_by_type[name]['I'].append(int(np.sum(states[mask]==1)))
                hist_by_type[name]['R'].append(int(np.sum(states[mask]==2)))
    return hist_full, hist_by_type


h_typed_full, h_typed = eval_marl_typed(marl_ba)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
type_colors = {'Informed': '#2196F3', 'Uninformed': '#FF5722', 'Indifferent': '#9E9E9E'}

# Full population
ax = axes[0, 0]
ax.plot(h_typed_full['S'], 'b-', label='S (all)')
ax.plot(h_typed_full['I'], 'r-', label='I (all)')
ax.plot(h_typed_full['R'], 'g-', label='R (all)')
ax.set_title('Full Population — MARL-DQN'); ax.legend(); ax.grid(alpha=0.3)

# Per-type infected curves
ax2 = axes[0, 1]
for name, col in type_colors.items():
    ax2.plot(h_typed[name]['I'], color=col, label=f'I — {name}', linewidth=2)
ax2.set_title('Infected Curves by Node Type')
ax2.set_xlabel('Time'); ax2.set_ylabel('Infected nodes')
ax2.legend(); ax2.grid(alpha=0.3)

# Per-type refuted curves  
ax3 = axes[1, 0]
for name, col in type_colors.items():
    ax3.plot(h_typed[name]['R'], color=col, label=f'R — {name}', linestyle='--', linewidth=2)
ax3.set_title('Refuted/Recovered Curves by Node Type')
ax3.set_xlabel('Time'); ax3.set_ylabel('Refuted nodes')
ax3.legend(); ax3.grid(alpha=0.3)

# Peak infected per type (bar)
ax4 = axes[1, 1]
peak_I = {name: max(h_typed[name]['I']) for name in type_colors}
ax4.bar(peak_I.keys(), peak_I.values(),
        color=list(type_colors.values()), edgecolor='black', alpha=0.85)
ax4.set_title('Peak Infected by Node Type')
ax4.set_ylabel('Peak I'); ax4.grid(axis='y', alpha=0.3)
for j, (name, val) in enumerate(peak_I.items()):
    ax4.text(j, val+0.5, str(val), ha='center', fontweight='bold')

fig.suptitle('Node-Type Breakdown: MARL-DQN Infection Dynamics', fontsize=13, fontweight='bold')
fig.tight_layout(); plt.show()
fig.savefig('outputs/19_nodetype_breakdown.png', dpi=110, bbox_inches='tight')

print('\nPeak infected by node type:')
for name, val in peak_I.items():
    print(f'  {name:15s}: {val}')
print('\nExpected: Uninformed > Informed > Indifferent')
print('Indifferent plateau visible as slow mid-outbreak growth in full-population I curve')


## Section 10 — Influential Efficiency & Rumour Generation/Detection

### Influential Efficiency
Measures whether MARL leverages high-PageRank hub nodes better than classical SIR
during the containment phase. Positive IE indicates MARL recovers influential nodes faster.

### Rumour Generation & Detection
Runs N independent outbreaks on the same trained graph. Each starts from a fresh random seed set.
A rumour is *detected* the first time the recovered fraction R(t)/n crosses 5% of all nodes.


In [ ]:
N_RUMOURS        = 10    # number of independent rumour outbreaks per condition
DETECT_THRESHOLD = 0.05  # detection triggered when recovered fraction exceeds this

# ── Influential Efficiency ────────────────────────────────────────────────
ie_results = plot_influential_efficiency(marl_ba.gc, h_marl, h_sir,
                                         fname='20_influential_efficiency')

# ── Rumour Generation & Detection: SIR baseline ───────────────────────────
print(f'\nSimulating {N_RUMOURS} SIR rumours...')
res_sir_rumour = simulate_multi_rumour(
    gc=marl_ba.gc, marl_system=None,
    n_rumours=N_RUMOURS, detection_threshold=DETECT_THRESHOLD, seed=SEED)

# ── Rumour Generation & Detection: MARL-DQN ──────────────────────────────
print(f'Simulating {N_RUMOURS} MARL-DQN rumours...')
res_marl_rumour = simulate_multi_rumour(
    gc=marl_ba.gc, marl_system=marl_ba,
    n_rumours=N_RUMOURS, detection_threshold=DETECT_THRESHOLD, seed=SEED + 1)

# ── Plots & summary ───────────────────────────────────────────────────────
plot_rumour_metrics(res_sir_rumour, res_marl_rumour, n_rumours=N_RUMOURS,
                    fname='21_rumour_detection')
print_rumour_table(res_sir_rumour, res_marl_rumour)

# ── Full metric summary ───────────────────────────────────────────────────
print('\n── Full Metric Summary ──────────────────────────────────────')
I_s_ = np.array(h_sir['I']);  R_s_ = np.array(h_sir['R'])
I_m_ = np.array(h_marl['I']); R_m_ = np.array(h_marl['R'])
print(f'Spread Reduction      : {(np.max(I_s_)-np.max(I_m_))/(np.max(I_s_)+1e-9):.3f}')
print(f'Polarization SIR      : {np.mean(glob_pol(h_sir)):.3f}')
print(f'Polarization MARL     : {np.mean(glob_pol(h_marl)):.3f}')
print(f'Truth Amplif. SIR     : {np.mean(R_s_[:30])/(np.mean(I_s_[:30])+1e-9):.3f}')
print(f'Truth Amplif. MARL    : {np.mean(R_m_[:30])/(np.mean(I_m_[:30])+1e-9):.3f}')
print(f'Influential Efficiency: {ie_results["IE"]:+.3f}')
print(f'Detection rate SIR    : {res_sir_rumour["detection_rate"]:.1%}')
print(f'Detection rate MARL   : {res_marl_rumour["detection_rate"]:.1%}')
